In [1]:
using PeriodicOrbitTTV
using PeriodicOrbitTTV: compute_system_init, orbital_to_cartesian, calculate_jac_time_evolution, extract_elements
using NbodyGradient

using FiniteDifferences
using Test

[ Info: Precompiling PeriodicOrbitTTV [39b8e391-62ca-4e8a-af42-0d151c329bad] (cache misses: wrong dep version loaded (2))
┌ Info: Skipping precompilation due to precompilable error. Importing PeriodicOrbitTTV [39b8e391-62ca-4e8a-af42-0d151c329bad].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.


In [2]:
optvec_0 = BigFloat.([0.007634091834320847, 0.015068095357529813, 0.0008875855342026987, -1.4454314813764126, -0.728172966274584, 2.7947084462295138, 3.147750653875983, -3.162598509555049, 0.00017275886406294735, 12.142001475526312, 4.264843854263061e-5, 2.379000115670915e-5, 7.420771108125184e-5, 2.004292806015084, 0.0003710381951454858, 48.46641242308062])

orbparams = OrbitParameters(3, BigFloat.([0.5]), BigFloat(500.))

nplanet = 3
epsilon = BigFloat(eps(Float64))

# ForwardDiff
orbit_0 = Orbit(nplanet, OptimParameters(nplanet, deepcopy(optvec_0)), orbparams)

Orbit
Number of planets: 3
Current time: 0.0


Planet masses: BigFloat[4.2648438542630611011903518647869759661261923611164093017578125e-05, 2.37900011567091491482099641086023211755673401057720184326171875e-05, 7.42077110812518398776094219471133328625001013278961181640625e-05]


In [28]:
function to_matrix(tt::Matrix{T}) where T <: Real
    tt_mat = Matrix{T}(undef, 0, 4)
    
    for i in 1:size(tt, 1), j in 1:size(tt, 2)
        if tt[i,j] != 0.0
            tt_mat = vcat(tt_mat, [i-1, j-1, tt[i,j], 0.]')
        end
    end
    return tt_mat
end

function log_prior(optvec::Vector{T}, truthvec::Vector{T}, weights_prior::Vector{T}) where T <: Real
    log_prior = -0.5 * sum(((optvec .- truthvec).^2 .* weights_prior))
    return log_prior
end

function log_likelihood(optvec::Vector{T}, truthvec::Vector{T},weights_periodic::Vector{T}, nplanet::Int, orbparams::OrbitParameters{T}, tt_data::Matrix{T}) where T <: Real
    log_lh = 0.0
    # PO Contribution
    orbit = Orbit(nplanet, OptimParameters(nplanet, optvec), orbparams)
    log_lh += -0.5 * sum((param_diff(3, orbit.final_elem, orbit.init_elem)[1:end-2]).^2 .* weights_periodic)
    
    # TT Contribution
    tt = compute_tt(orbit, orbparams.obstmax)
    tt_mat = to_matrix(tt.tt)
    log_lh += -0.5 * sum(((tt_mat[:,3] .- tt_data[:,3])./tt_data[:,4]).^2)
    
    return log_lh
end

function log_probability(optvec::Vector{T}, truthvec::Vector{T}, weights_prior::Vector{T}, weights_periodic::Vector{T}, nplanet::Int, orbparams::OrbitParameters{T}, tt_data::Matrix{T}) where T <: Real
    # Scale the optvec before use
    
    log_p = log_prior(optvec, truthvec, weights_prior)
    return log_p + log_likelihood(optvec, truthvec, weights_periodic, nplanet, orbparams, tt_data)
end

function residues_grad(optvec::Vector{T}, truthvec::Vector{T},weights_periodic::Vector{T}, nplanet::Int, orbparams::OrbitParameters{T}, tt_data::Matrix{T}) where T <: Real
    residues = zeros(T, 9nplanet - 1 + size(tt_data, 1))
    
    # PO Contribution
    orbit = Orbit(nplanet, OptimParameters(nplanet, optvec), orbparams)
    residues[1:4nplanet-2] = 2 * (param_diff(3, orbit.final_elem, orbit.init_elem)[1:end-2]) .* weights_periodic

    # Prior contribution
    residues[4nplanet-1:9nplanet-1] .= 0
    
    # TT Contribution
    tt = compute_tt(orbit, orbparams.obstmax)
    tt_mat = to_matrix(tt.tt)
    residues[9nplanet:end] = 2 * ((tt_mat[:,3] .- tt_data[:,3]) ./ (tt_data[:,4].^2))
    
    return -0.5 * residues
end

residues_grad(θ) = residues_grad(θ, optvec_0, fill(po_weight, 10), 3, orbparams, tt_data)

using PeriodicOrbitTTV: compute_diff_squared_jacobian

# Wrapper for log-likelihood
llhood(theta) = log_probability(theta .* mc_scale_factor, optvec_0, fill(0., 16), fill(po_weight, 10), 3, orbparams, tt_data)

# Wrapper for grad computation
function llhood_grad(θ::Vector{<:Real})
    log_llhood = let mc_scale_factor = mc_scale_factor
        llhood(θ)
    end
        
    grad = let nplanet = 3, mc_scale_factor = mc_scale_factor
        begin
            optparams = OptimParameters(nplanet, θ .* mc_scale_factor)
            jac = compute_diff_squared_jacobian(optparams, orbparams, nplanet, tt_data)

            vec(permutedims(residues_grad(θ .* mc_scale_factor)' * jac)) .* mc_scale_factor
        end
    end
    return log_llhood, grad
end

po_weight = BigFloat.(1490.)
mc_scale_factor = 10 .^ round.(log10.(abs.(optvec_0)))

16-element Vector{Float64}:
   0.010000000000000002
   0.010000000000000002
   0.001
   1.0
   1.0
   1.0
   1.0
  10.0
   0.0001
  10.0
   0.0001
   1.0e-5
   0.0001
   1.0
   0.001
 100.0

In [15]:
using DelimitedFiles

TT_SOURCE = "sample_orbit_v1_TT_30secs.in"
tt_data = readdlm(TT_SOURCE,'\t',comments=true,comment_char='#');
tt_data = convert(Matrix{BigFloat}, tt_data);

In [16]:
fd_grad = grad(central_fdm(2,1), llhood, optvec_0 ./ mc_scale_factor)[1]

16-element Vector{BigFloat}:
     309.5554591201810953211171439857969001904770076150737448938160197612763470279011
    1814.756372471503083536457640008291004036395948723810401638904958149611363015914
      36.90054804344993538616400102710998662466096274264866570415149287108136759981259
  -14326.78619828192575687341369583682434286444819673911280711149247256458225013232
   93782.22323694314143046138413375160453680849341182825478227247484273590202353758
  -20306.29442328933030508590297855109029758475647564480673064093647001268387897209
   73242.73959229682898995264091715506837971035558753435601206194019669029931887006
 -203119.1050759986091243467115861535453860326245744722976452215979503160270499711
      36.34091241146358218020651240790872043024894122249494715652104457307232636947586
 -440734.0138637761214339911933117258229962596260752234479888085352184801749380338
    -481.3281024393924656214320526524958002188993229262395254477361713493696835275252
     -66.34899242205151717258111566913

In [19]:
ad_grad = llhood_grad(optvec_0 ./ mc_scale_factor)[2]

16-element Vector{BigFloat}:
     309.555459120181095338247069879172337932022523121100010050964133028953525983053
    1814.756372471503083481303362082457726083597728438578117579660655523839425750125
      36.90054804344993441956099742512139000594108037537585220480494676225087852779987
  -14326.78619828192575687194682799444410622969193603783989022634883333797487797212
   93782.22323694314143036781475330252763505208541616663954745915903427020252867918
  -20306.29442328933030745580376788273539019869973498512713827852438555365802870708
   73242.73959229682898995264728948826217202926933194343574096139308645391376760551
 -203119.1050759986091243467086260133622552168155102221009548533800723218948250036
      36.34091241146359029367962157199786134833321604555718607995693064413919160356627
 -440734.0139049264448984417224072403645003384318479621804113754801738249439415377
    -481.3281024393924292794549508298868430288303189004097447253390509273798698177841
     -66.348992422051521967079527764556

In [20]:
fd_grad ./ ad_grad

16-element Vector{BigFloat}:
 0.9999999999999999999446628208655659503641948678478654716562767586391086446753277
 1.00000000000000000003039211144949395096591753319575898589770886093417085038592
 1.000000000000000026194814300964463538840544855343623962197600335990904483819636
 1.000000000000000000000102386384641947406533964485288623450633696190494935768592
 1.000000000000000000000997730457004324934513940009712691443176059553912142629691
 0.9999999999999999998832923063198768030726383086849227284501946366135796526647898
 0.9999999999999999999999999129970666135143122363999259769947553520003183546640103
 1.00000000000000000000000001457342076228258341797487383959261864806561412361305
 0.9999999999999997767399723677623554881764413712290377693582643735163531493502319
 0.9999999999066322948395643256234104279168114133710237442091056594609783683153646
 1.000000000000000075503543046084028060687198494200252391759579839824837298162196
 0.999999999999999927738188070781656597307890991693537474857260979

In [21]:
# Normal Precision

In [29]:
optvec_0 = ([0.007634091834320847, 0.015068095357529813, 0.0008875855342026987, -1.4454314813764126, -0.728172966274584, 2.7947084462295138, 3.147750653875983, -3.162598509555049, 0.00017275886406294735, 12.142001475526312, 4.264843854263061e-5, 2.379000115670915e-5, 7.420771108125184e-5, 2.004292806015084, 0.0003710381951454858, 48.46641242308062])

orbparams = OrbitParameters(3, ([0.5]), (500.))

nplanet = 3
epsilon = (eps(Float64))

# ForwardDiff
orbit_0 = Orbit(nplanet, OptimParameters(nplanet, deepcopy(optvec_0)), orbparams)

po_weight = (1490.)
mc_scale_factor = 10 .^ round.(log10.(abs.(optvec_0)))

TT_SOURCE = "sample_orbit_v1_TT_30secs.in"
tt_data = readdlm(TT_SOURCE,'\t',comments=true,comment_char='#');
# tt_data = convert(Matrix{BigFloat}, tt_data);

In [30]:
ad_grad_f64 = llhood_grad(optvec_0 ./ mc_scale_factor)[2]

16-element Vector{Float64}:
     309.5554583190094
    1814.7563741611145
      36.90054826395998
  -14326.786153400995
   93782.22333129415
  -20306.29454790445
   73242.73956090977
 -203119.1063223609
      36.34091265742231
 -440734.0230930049
    -481.32810351274685
     -66.34899280290814
     249.10324424570013
 -493787.8542082978
      59.183808006013706
       0.0380802871477385

In [31]:
fd_grad_f64 = grad(central_fdm(2,1), llhood, optvec_0 ./ mc_scale_factor)[1]

16-element Vector{Float64}:
     310.72217823074874
    1819.6135053782016
      37.30734238756938
  -13987.696682635566
   93286.51147184115
  -20447.40879709001
   73695.49071470431
 -203065.08436017318
      36.5608750423816
 -440890.9082483224
    -475.31759489792347
     -71.32513708772854
     249.1196356121129
 -493779.34068223974
      59.42836193304421
      -0.09531523476063687

In [32]:
fd_grad_f64 ./ ad_grad_f64

16-element Vector{Float64}:
  1.0037690174099174
  1.0026764646132362
  1.0110240671954107
  0.9763317838952363
  0.9947142236359459
  1.0069492860380145
  1.006181515826808
  0.9997340380077195
  1.0060527479602075
  1.0003559633409205
  0.987512658058072
  1.0749995452018721
  1.0000658014971358
  0.9999827587374102
  1.0041321086842816
 -2.5030072486283084